In [2]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import re
import pickle

# List of FNO Stocks

In [ ]:
#Run periodically to update symbols and stocks list
'''
fno_list = pd.read_csv('./Cache/fno_stocks.csv')

drop_words = [' Limited', ' Ltd', ' Industries', 'The ', ' (India)', ' (india)',' Enterprises',' Enterprise', ' Company', ' Laboratories', ' Corporation']
fno_list['Stock'] = fno_list['Stock Name']

for word in drop_words:
    fno_list['Stock'] = fno_list['Stock'].map(lambda x: x.replace(word, ''))
    
fno_list['Stock'] = fno_list['Stock'].map(lambda x: x.lower())
fno_list['Symbol'] = fno_list['Symbol'].map(lambda x: x.lower())

symbols = fno_list['Symbol'].values
stocks = fno_list['Stock'].values

with open('./Cache/symbols.pkl', 'wb') as f:
    pickle.dump(symbols, f)
with open('./Cache/stocks.pkl', 'wb') as f:
    pickle.dump(stocks, f)
'''

In [ ]:
#read symbols and stocks from pickle files
with open('./Cache/symbols.pkl', 'rb') as f:
    symbols = pickle.load(f)
with open('./Cache/stocks.pkl', 'rb') as f:
    stocks = pickle.load(f)

filter = sorted(list(set(list(symbols) + list(stocks))))


# Request Data from Pulse

In [5]:
# Function: fetch_pulse_elements
def fetch_pulse_elements(url='https://pulse.zerodha.com', headers=None, parser='html.parser', timeout=10):
    """Request Pulse homepage and return soup and element lists: headlines, descriptions, sources, dates."""
    if headers is None:
        headers = {
            'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2.1 Safari/605.1.15'
        }
    resp = requests.get(url, headers=headers, timeout=timeout)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, parser)
    # Remove 'similar' article blocks that are duplicates
    for ul in soup.select('ul.similar'):
        ul.decompose()
    headlines = soup.select('h2.title')
    descriptions = soup.select('div.desc')
    sources = soup.select('span.feed')
    dates = soup.select('span.date')
    return soup, headlines, descriptions, sources, dates

In [6]:
# Function: build_news_dataframe
from urllib.parse import urljoin
def build_news_dataframe(headlines, descriptions, sources, dates, base_url='https://pulse.zerodha.com'):
    """Convert raw element lists from Pulse into a normalized DataFrame with absolute links."""
    headlines_list = [h.get_text(strip=True).lower() for h in headlines]
    source_links = []
    for h in headlines:
        a = h.find('a')
        href = a.get('href') if a else ''
        href = urljoin(base_url, href)
        source_links.append(href)
    descriptions_list = [d.get_text(strip=True).lower() for d in descriptions]
    sources_list = [s.get_text(strip=True).lower().replace('— ', '') for s in sources]
    dates_list = [d.get('title', '') for d in dates]
    df = pd.DataFrame({
        'Date': dates_list,
        'Headlines': headlines_list,
        'Description': descriptions_list,
        'Source': sources_list,
        'Source_Link': source_links
    })
    return df

# Filter Articles for FNO Stocks

In [7]:
# Function: assign_tags
def assign_tags(df, tag_list):
    """Assign a comma-separated 'Tags' column when items from tag_list are present in Description or Headlines."""
    import re as _re
    # normalize tag list to lowercase for robust matching
    tag_list_lower = [t.lower() for t in tag_list]
    def _find_tags_row(row):
        # combine description and headlines, handle missing values
        desc = '' if row.get('Description') is None else str(row.get('Description'))
        head = '' if row.get('Headlines') is None else str(row.get('Headlines'))
        text = (desc + ' ' + head).lower()
        matches = [t for t in tag_list_lower if _re.search(r'\b{}\b'.format(_re.escape(t)), text)]
        return ', '.join(matches)
    df = df.copy()
    df['Tags'] = df.apply(_find_tags_row, axis=1)
    return df

In [8]:
# Function: postfilter_articles (drop empty tags)
def postfilter_articles_drop_empty(df):
    df = df.copy()
    df = df[df['Tags'] != '']
    df = df.reset_index(drop=True)
    return df

In [9]:
# Function: postfilter_articles_max_tags
def postfilter_articles_max_tags(df, max_tags=3):
    df = df.copy()
    # Count non-empty tag tokens (comma-separated)
    def _count_tags(s):
        if not s: return 0
        parts = [p.strip() for p in s.split(',') if p.strip()!='']
        return len(parts)
    df['Num_Tags'] = df['Tags'].apply(_count_tags)
    df = df[df['Num_Tags'] <= max_tags].reset_index(drop=True)
    return df

# Get Detailed News Article

In [10]:
# Call the helper functions in sequence to build news_df (functions defined above)
base_url = 'https://pulse.zerodha.com'
request_headers = {
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2.1 Safari/605.1.15'
}
# 1) Fetch elements from Pulse
soup, headlines, descriptions, sources, dates = fetch_pulse_elements(base_url, headers=request_headers)

# 2) Build dataframe from elements
news_df = build_news_dataframe(headlines, descriptions, sources, dates, base_url=base_url)

# 3) Assign tags from the 'filter' list (from FNO list cell)
news_df = assign_tags(news_df, filter)

# 4) Post-filter: drop empty tags, then drop articles with too many tags
news_df = postfilter_articles_drop_empty(news_df)
news_df = postfilter_articles_max_tags(news_df, max_tags=3)

# Ready for the next step (detailed article scraping)
news_df.head()

,Date,Headlines,Description,Source,Source_Link,Tags,Num_Tags
0,"11:53 AM, 29 Dec 2025","7 long weekends! nse, bse 2026 stock market ho...",nse and bse announce the 2026 trading holiday ...,economic times,https://economictimes.indiatimes.com/markets/s...,bse,1
1,"11:35 AM, 29 Dec 2025",silver futures jump 6% to record ₹2.54 lakh/kg...,"meanwhile, gold futures hovered near all-time ...",the hindu business,https://www.thehindu.com/business/markets/silv...,mcx,1
2,"11:19 AM, 29 Dec 2025","encora buyout widens healthcare, high-tech pla...",coforge's acquisition of encora significantly ...,economic times,https://economictimes.indiatimes.com/markets/e...,coforge,1
3,"11:14 AM, 29 Dec 2025",vedanta shares gain over 2% to hit 52-week hig...,vedanta shares hit a fresh 52-week high as ris...,economic times,https://economictimes.indiatimes.com/markets/s...,"hindustan zinc, vedanta",2
4,"10:43 AM, 29 Dec 2025",stock markets gather momentum after muted begi...,the 30-share bse sensex went up by 22.24 point...,the hindu business,https://www.thehindu.com/business/markets/stoc...,"bse, nifty",2


# Get Detailed Article Text

In [11]:
from newspaper import Article

def extract_with_newspaper(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        text = article.text.strip()
        if len(text) > 500:
            return text
    except Exception:
        pass
    return None

In [12]:
from readability.readability import Document
from bs4 import BeautifulSoup
import requests

def extract_with_readability(url):
    try:
        html = requests.get(url, timeout=10, headers={
            "User-Agent": "Mozilla/5.0"
        }).text

        doc = Document(html)
        soup = BeautifulSoup(doc.summary(), "html.parser")

        paragraphs = [
            p.get_text(" ", strip=True)
            for p in soup.find_all("p")
        ]

        text = "\n".join(paragraphs)
        if len(text) > 500:
            return text
    except Exception:
        pass

    return None

In [13]:
def is_valid_article(text):
    if text is None:
        return False
    if len(text.split()) < 150:
        return False
    if "subscribe" in text.lower():
        return False
    return True

In [14]:
def get_article_text(url):
    text = extract_with_newspaper(url)
    if is_valid_article(text):
        return text

    text = extract_with_readability(url)
    if is_valid_article(text):
        return text

    return None

In [15]:
news_df['Complete_Article'] = ''

for i in news_df.index:
    news_df.loc[i, 'Complete_Article'] = get_article_text(news_df.loc[i, 'Source_Link'])

In [16]:
#Filtering out errors and paid articles
news_df = news_df[news_df['Complete_Article']!='Error ocurred']
news_df = news_df[news_df['Complete_Article']!='']
news_df.head()

,Date,Headlines,Description,Source,Source_Link,Tags,Num_Tags,Complete_Article
0,"11:53 AM, 29 Dec 2025","7 long weekends! nse, bse 2026 stock market ho...",nse and bse announce the 2026 trading holiday ...,economic times,https://economictimes.indiatimes.com/markets/s...,bse,1,The National Stock Exchange (NSE) has released...
1,"11:35 AM, 29 Dec 2025",silver futures jump 6% to record ₹2.54 lakh/kg...,"meanwhile, gold futures hovered near all-time ...",the hindu business,https://www.thehindu.com/business/markets/silv...,mcx,1,Silver prices extended their record-breaking r...
2,"11:19 AM, 29 Dec 2025","encora buyout widens healthcare, high-tech pla...",coforge's acquisition of encora significantly ...,economic times,https://economictimes.indiatimes.com/markets/e...,coforge,1,’s acquisition of Encora marks a major strateg...
3,"11:14 AM, 29 Dec 2025",vedanta shares gain over 2% to hit 52-week hig...,vedanta shares hit a fresh 52-week high as ris...,economic times,https://economictimes.indiatimes.com/markets/s...,"hindustan zinc, vedanta",2,Shares of metal and mining major\nsurged 2.47%...
4,"10:43 AM, 29 Dec 2025",stock markets gather momentum after muted begi...,the 30-share bse sensex went up by 22.24 point...,the hindu business,https://www.thehindu.com/business/markets/stoc...,"bse, nifty",2,Stock market benchmark indices Sensex and Nift...


In [17]:
#Exporting
news_df.to_pickle("./Data/Pulse/pulse_news.pkl")